# 04. PyTorch nn.Module Basics - 练习

**难度：** Medium | **标签：** `PyTorch`, `nn.Module`, `Parameters` | **目标人群：** PyTorch 入门学习者

本练习配套导学：[Chapter 0 导学](./intro.md)

## 🎯 学习目标
- 掌握 `nn.Module` 的定义方式
- 理解参数注册与 `state_dict`
- 学会组合简单模块搭建 MLP


## Part 1: 定义基础模块

### 练习 1.1: 手写线性层
实现一个不依赖 `nn.Linear` 的线性层，显式定义权重和偏置。


In [ ]:
import torch
import torch.nn as nn


class SimpleLinear(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        self.weight = nn.Parameter(torch.empty(out_features, in_features))
        self.bias = nn.Parameter(torch.zeros(out_features))
        nn.init.xavier_uniform_(self.weight)

    def forward(self, x):
        return x @ self.weight.t() + self.bias


class TwoLayerMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        self.fc1 = SimpleLinear(input_dim, hidden_dim)
        self.act = nn.GELU()
        self.fc2 = SimpleLinear(hidden_dim, output_dim)

    def forward(self, x):
        return self.fc2(self.act(self.fc1(x)))


def count_parameters(module):
    return sum(param.numel() for param in module.parameters())


In [ ]:
def test_simple_linear():
    layer = SimpleLinear(2, 1)
    with torch.no_grad():
        layer.weight.copy_(torch.tensor([[2.0, -1.0]]))
        layer.bias.copy_(torch.tensor([0.5]))
    x = torch.tensor([[3.0, 4.0]])
    y = layer(x)
    assert torch.allclose(y, torch.tensor([[2.5]]))
    print('✅ SimpleLinear 通过')


def test_two_layer_mlp():
    model = TwoLayerMLP(4, 8, 2)
    x = torch.randn(3, 4)
    y = model(x)
    assert y.shape == (3, 2)
    expected_params = (4 * 8 + 8) + (8 * 2 + 2)
    assert count_parameters(model) == expected_params
    print('✅ TwoLayerMLP 通过')


def test_state_dict_roundtrip():
    model = TwoLayerMLP(3, 6, 2)
    before = {k: v.clone() for k, v in model.state_dict().items()}
    buffer = model.state_dict()
    model.load_state_dict(buffer)
    after = model.state_dict()
    for key in before:
        assert torch.allclose(before[key], after[key])
    print('✅ state_dict roundtrip 通过')


test_simple_linear()
test_two_layer_mlp()
test_state_dict_roundtrip()


## Part 2: 实战应用

### 练习 2.1: 观察参数管理
列出模型所有参数名称和形状，并对比 `state_dict()` 的内容。


In [ ]:
model = TwoLayerMLP(4, 8, 2)
print('参数总量:', count_parameters(model))
print('named_parameters():')
for name, param in model.named_parameters():
    print(f'  {name}: {tuple(param.shape)}')

print()
print('state_dict keys:')
for key in model.state_dict().keys():
    print(' ', key)
